In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerForecaster(nn.Module):
    def __init__(self, in_ch, n_cities, d_model, heads, layers, ff_mult, head_hidden, dropout, emb_dim=8, out_dim=3):
        super().__init__()
        self.proj = nn.Linear(in_ch, d_model)
        self.pos = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, heads, dim_feedforward=ff_mult * d_model, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, norm=nn.LayerNorm(d_model), enable_nested_tensor=False)
        self.emb = nn.Embedding(n_cities, emb_dim)
        self.head = nn.Sequential(nn.Linear(d_model + emb_dim, head_hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(head_hidden, out_dim))

    def forward(self, x, city):
        z = self.encoder(self.pos(self.proj(x)))
        return self.head(torch.cat([z.mean(dim=1), self.emb(city)], dim=1))


